In [7]:
import os
import shutil
from MRE import Job # Importamos el emulador

# Etiquetas para identificar de qué dataset viene cada equipo
TAG_PLATINIUM = "PLAT"
TAG_PREMIUM = "PREM"

# --- FASE MAP ---
# Se ejecuta por cada línea. key = ID local, value = "ID_visitante ap_local ap_vis"
# Hay un map definido por dataset para etiquetar el origen (PLAT / PREM).
# Se FILTRA en el map: solo emitimos al local si tuvo más de X veces las apuestas del visitante.

# Filtramos el dataset Platinium
def fmapPlatinium(key, value, context):
    X = context["X"]  # parámetro de la consulta
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_visitante = datos[0]  # ID_equipo_visitante
    ap_local = int(datos[1])  # usuarios que apostaron al local como ganador
    ap_visitante = int(datos[2])  # usuarios que apostaron al visitante como ganador

    # Solo emitimos si el local superó X veces las apuestas del visitante
    if ap_local > X * ap_visitante:
        context.write(id_local, TAG_PLATINIUM)

# Filtramos el dataset Premium
def fmapPremium(key, value, context):
    X = context["X"]  # parámetro de la consulta
    datos = value.split() # separa la línea en campos
    id_local = key  # ID_equipo_local es la clave
    id_visitante = datos[0]  # ID_equipo_visitante
    ap_local = int(datos[1])  # usuarios que apostaron al local como ganador
    ap_visitante = int(datos[2])  # usuarios que apostaron al visitante como ganador

    # Solo emitimos si el local superó X veces las apuestas del visitante
    if ap_local > X * ap_visitante:
        context.write(id_local, TAG_PREMIUM)

# --- FASE COMBINER ---
# Elimina etiquetas repetidas por equipo, solo con lo que vio su mapper.
# Reduce el tráfico de red. NO decide la diferencia (ya que solo ve datos parciales).
def fcombiner(key, values, context):
    # set() deja una sola vez cada etiqueta; emitimos el mismo formato que el map
    for tag in set(values):
        context.write(key, tag)

# --- FASE REDUCE ---
# Recibe todas las etiquetas de UN equipo (key) ya agrupadas por el shuffle.
# Acá se decide la diferencia.
def fred(key, values, context):
    tags = set(values) # etiquetas distintas que llegaron para este equipo

    # Diferencia (Platinium - Premium): cumple en platinium y NO en premium
    if TAG_PLATINIUM in tags and TAG_PREMIUM not in tags:
        context.write(key, "")

# --- DRIVER ---
# Configura y dispara el job
if __name__ == '__main__':
    input_plat = "/content/in_platinium"   # carpeta con el dataset platinium
    input_prem = "/content/in_premium"     # carpeta con el dataset premium
    output_dir = "/content/salida_punto3"  # carpeta de resultados

    # Preparación: crear carpetas de entrada y copiar los archivos
    os.makedirs(input_plat, exist_ok=True)
    os.makedirs(input_prem, exist_ok=True)
    shutil.copy("/content/apuestasPlatinium.txt", input_plat + "/apuestasPlatinium.txt")
    shutil.copy("/content/apuestasPremium.txt", input_prem + "/apuestasPremium.txt")

    # Borrar la salida anterior (si existe) para que no falle al re-ejecutar
    shutil.rmtree(output_dir, ignore_errors=True)

    job3 = Job(input_plat, output_dir, fmapPlatinium, fred) # Job con el primer dataset platinium y su map correspondiente
    job3.addInputPath(input_prem, fmapPremium) # Segundo dataset premium con su map
    job3.setCombiner(fcombiner) # Combiner para reducir datos enviados por la red
    job3.setParams({"X": 2}) # Parámetro X ajustable a la consulta

    success = job3.waitForCompletion()

    if success:
        print("El job finalizó correctamente!")

El job finalizó correctamente!
